<a href="https://colab.research.google.com/github/liudi23/determine/blob/main/Week%202/1_st445_week2_lab1_api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ST445 Week 2 seminar, lab 1: collecting data from APIs

In this lab you build API requests. You then learn the routine professionals follow for every request, turn the answers into tables, and finish with research tasks that use the same skills your project will need.

The main source is the **World Bank API**, which publishes economic and social indicators for every country, the **Open-Meteo** (weather), and the **Our World in Data chart API**.

### How the lab is organised

| Part | What happens | Time |
|---|---|---|
| **A. Guided demo** | Your tutor runs these cells with you. Run each one when they do, and paste the printed URLs into your browser. | 20 min |
| **B. Pause and analyse** | Short exercises: change one thing, predict the result, complete small pieces of code. | 20 min |
| **C. Two APIs, one dataset** | Collect from the Our World in Data and World Bank APIs, join on country and year, and validate the result: the same pipeline your project uses. | 10 min in the seminar, then after |

Cell marks: 🟢 run and read · 🟡 fill the gaps marked `____` · 🔵 write the code yourself · ✍️ write a short answer in the Markdown cell. For the exercises, you will find test cells to print ✅ or ❌ with a hint if got it correctly. Boxes titled **Method card** explain each new function: what it does and the minimum you need to set.

**Before you start this lab, save a copy in your local Google Drive at:** **File → Save a copy in Drive**, and rename it `<yourname>_week2_api_lab.ipynb`.

## Set-up

We use `requests` to send web requests; `pandas` to makes tables; the rest are small tools from Python's standard library. There are two helper functions at the bottom of some cells to `check` if you completed the exercises by printing: ✅ or ❌, and `filled` stops a cell if you have not replaced a `____` yet as required by the exercise.

In [ ]:
# 🟢 Run this first
import json, time, hashlib
from datetime import datetime, timezone
from pathlib import Path

import requests
import pandas as pd

HEADERS = {"User-Agent": "ST445 seminar lab (London School of Economics)"}

def check(condition, hint="Not yet. Read the task again."):
    print("✅ Correct" if condition else f"❌ {hint}")

def filled(*values):
    """False (with a reminder) if any value still contains ____."""
    if any("____" in str(v) for v in values):
        print("✋ Replace every ____ first, then run this cell again.")
        return False
    return True

print("Ready.")

---
# Part A. Guided demo

## A1. Building a request, one piece at a time

Every API request is a web address made of three pieces:

| Piece | Meaning | World Bank example |
|---|---|---|
| **Base URL** | where the API lives; the same for every request | `https://api.worldbank.org/v2` |
| **Endpoint** | *what* you are asking for; a path added to the base | `/country/GBR/indicator/SP.POP.TOTL` |
| **Parameters** | *how* you want it; `?` then `name=value` pairs joined by `&` | `?format=json&per_page=5` |

In the endpoint, `GBR` is the United Kingdom's three-letter ISO 3166 code (a  standard introduced in Week 1), and `SP.POP.TOTL` is the [World Bank's code for total population.](https://databank.worldbank.org/metadataglossary/subnational-population/series/SP.POP.TOTL) [Click here for a full list of indicators](https://data.worldbank.org/indicator) or [here for the Metadata Glossary](https://databank.worldbank.org/metadataglossary/all/series).

### Step 1: base URL plus endpoint

Run the cell, then **copy the printed URL into a new browser tab** and look at what comes back.

In [ ]:
# 🟢 Step 1
BASE = "https://api.worldbank.org/v2"
endpoint = "/country/GBR/indicator/SP.POP.TOTL"

url = BASE + endpoint
print(url)

**What you should see in the browser:** a page of **XML**, a tag-based format like HTML. The World Bank sends XML unless you ask for something else. The numbers are there, but XML tags can be more complex to work with in Python. Therefore, lets add a parameter to our API request, our first parameter, for a more familiar format.

### Step 2: add a parameter, `format=json`

Query Parameters start with `?`. Paste the new URL into your browser.

In [ ]:
# 🟢 Step 2
url = BASE + endpoint + "?format=json"
print(url)

**What you should see:** the same data as **JSON**. It starts with `[{"page":1,"pages":2,"per_page":50,"total":65,...}` and continues with a list of records, one per year, newest first. (Your numbers may differ slightly as new years are published.) By default you get **50 records per page**.

### Step 3: a second parameter, `per_page=5`

Extra parameters are joined with `&`. Predict what will change before you paste it.

In [ ]:
# 🟢 Step 3
url = BASE + endpoint + "?format=json&per_page=5"
print(url)

**What you should see:** only **5 records**, and the first part now says `"per_page":5` and many more `"pages"`. Same data, cut into smaller pages.

### Step 4: a third parameter, `date=2015:2020`

The [documentation](https://datahelpdesk.worldbank.org/knowledgebase/topics/125589-developer-information) says `date` takes a single year or a range written `start:end`.

In [ ]:
# 🟢 Step 4
url = BASE + endpoint + "?format=json&per_page=5&date=2015:2020"
print(url)

**What you should see:** 6 years (2015 to 2020), so `"total":6` and, with 5 per page, `"pages":2`. Note that year 2015 will not be displayed because you limited the number of elements to 5. Hence, data will be on the second page as well, unless you increase `per_page=6` instead.

Therefore, as you can observe, each parameter changed one thing about the answer obtained from the API.

### Step 5: let `requests` build the URL for you

Typing `?` and `&` by hand is error-prone. In practice you put the parameters in a **dictionary** and let `requests` build the URL. The cell below builds exactly the same URL as Step 4, without sending anything yet, so you can compare. [For more information on requests, click here.](https://requests.readthedocs.io/en/latest/api/)

In [ ]:
# 🟢 Step 5
params = {"format": "json", "per_page": 5, "date": "2015:2020"}
prepared = requests.Request("GET", BASE + endpoint, params=params).prepare()
print(prepared.url)
print("Same as Step 4:", prepared.url == BASE + endpoint + "?format=json&per_page=5&date=2015:2020")

From now on, every cell prints the URL it is about to request, so you can always paste it into a browser and see the raw answer for yourself.

For the remainder of this tutorial, we will use the small helper below that does exactly that: it will build the full URL of the API request to check its raw information on your browser.

In [ ]:
# 🟢 A helper you will use all lab
def show_url(base, endpoint, params=None):
    """Print and return the full URL that requests would send."""
    full = requests.Request("GET", base + endpoint, params=params).prepare().url
    print(full)
    return full

show_url(BASE, "/country/FRA/indicator/SP.POP.TOTL", {"format": "json", "per_page": 3})

## A2. The professional routine for every request

Getting *a* response is easy. Getting a response you can **trust** takes the same five steps every time. Professionals do them in this order, because each step protects the next:

| Step | What you do | Why |
|---|---|---|
| 1 | **Build and print the URL** | You can check it by eye and paste it into a browser |
| 2 | **Send the request with a timeout** | A slow server cannot freeze your notebook |
| 3 | **Check the status code before anything else** | 200 means the server handled it; 4xx means your request is wrong; 5xx means the server has a problem |
| 4 | **Check the body has the shape the documentation promises** | Some APIs report errors inside a 200 response (you will see one in a moment) |
| 5 | **Only then use the data, and save the raw response** | You never analyse an error message by mistake, and you can always go back to what you received |

> **Method card: `requests.get(url, params=None, headers=None, timeout=None)`**
>
> * `url` (**required**): base plus endpoint, without the `?...` part.
> * `params`: a dictionary of parameters; `requests` adds `?` and `&`.
> * `headers`: a dictionary of extra information; we send a `User-Agent` saying who we are.
> * `timeout`: seconds to wait before giving up. **Always set it**; without it, a request can hang for ever.
>
> It returns a **response** object `r`. The parts you need: `r.status_code` (a number such as 200), `r.url` (the URL actually sent), `r.headers["Content-Type"]` (what kind of data came back), and `r.json()` (the body turned into Python lists and dictionaries; only call it once you know the body is JSON).

Here are the five steps written out, one line each, for population in the UK:

In [ ]:
# 🟢 The five steps, one at a time
params = {"format": "json", "per_page": 5, "date": "2015:2020"}

# 1. Build and print the URL
show_url(BASE, endpoint, params)

# 2. Send it, with a timeout
r = requests.get(BASE + endpoint, params=params, headers=HEADERS, timeout=30)

# 3. Check the status code first
print("Status code:", r.status_code)
if r.status_code != 200:
    raise RuntimeError(f"Request failed with status {r.status_code}")

# 4. Check the body is the shape the documentation promises: a list of [metadata, records]
body = r.json()
print("Content type:", r.headers.get("Content-Type"))
print("Body is a list of length", len(body))

# 5. Only now use the data
metadata, records = body[0], body[1]
print("Metadata:", metadata)
print("First record:", records[0])

**What you should see:** status **200**; content type `application/json`; a list of length **2**. The first element is the **metadata** (page numbers and totals); the second is the list of **records**. A record looks like:

```
{"indicator": {"id": "SP.POP.TOTL", "value": "Population, total"},
 "country": {"id": "GB", "value": "United Kingdom"},
 "countryiso3code": "GBR", "date": "2020", "value": 67081234, ...}
```

Two Week 1 ideas are visible: `date` is **text**, not a number; and `indicator` and `country` are small dictionaries **nested** inside each record. Therefore, the data type is not the expected as date should be int or float.

### When step 4 matters: an error hidden in a 200

Now ask for a country that does not exist, writing the name instead of the code. Predict the status code first.

In [ ]:
# 🟢 Predict, then run
bad_params = {"format": "json"}
show_url(BASE, "/country/France/indicator/SP.POP.TOTL", bad_params)
r = requests.get(BASE + "/country/France/indicator/SP.POP.TOTL", params=bad_params,
                 headers=HEADERS, timeout=30)
print("Status code:", r.status_code)
print("Body:", r.text[:300])

**What you should see:** status **200**, yet the body is an error message:

```
[{"message": [{"id": "120", "key": "Invalid value", "value": "The provided parameter value is not valid"}]}]
```

**Error 120** means one of the values in your URL is not valid: here, `France` is not a country code. (The World Bank accepts `FRA` or `FR`.) The server *handled* the request, so it says 200, but you did not get data. That is why step 4 exists. If you meet error 120 later in this lab, check the spelling of every code in your URL.

### Putting the routine into one function

Rather than repeat five steps in every cell, professionals wrap them in a function. The function below also saves each response in a `cache/` folder and uses the saved copy if the network fails, which keeps a busy seminar room working.

In [ ]:
# 🟢 The routine as a function
CACHE = Path("cache"); CACHE.mkdir(exist_ok=True)

def get_worldbank(endpoint, params, name):
    """Steps 1 to 5 for the World Bank API. Returns (metadata, records)."""
    show_url(BASE, endpoint, params)                                        # 1
    path = CACHE / f"{name}.json"
    try:
        r = requests.get(BASE + endpoint, params=params, headers=HEADERS, timeout=30)   # 2
        if r.status_code != 200:                                            # 3
            raise RuntimeError(f"Status {r.status_code}: {r.text[:200]}")
        body = r.json()
        path.write_bytes(r.content)                                         # 5: keep the raw response
    except requests.RequestException as err:
        if not path.exists():
            raise
        print(f"(Network problem: {type(err).__name__}. Using the saved copy.)")
        body = json.loads(path.read_text(encoding="utf-8"))
    if isinstance(body, list) and body and "message" in body[0]:            # 4
        msg = body[0]["message"][0]
        raise ValueError(f"World Bank error {msg['id']}: {msg['value']}. Check every code in the URL.")
    return body[0], body[1] or []

metadata, records = get_worldbank(endpoint, {"format": "json", "per_page": 50}, "gbr_population_p1")
print(f"Page {metadata['page']} of {metadata['pages']}, {len(records)} records here, {metadata['total']} in total")

## A3. Getting every page

With 50 records per page and 65 years of data, the answer comes in **2 pages**. `metadata["pages"]` tells you how many, and the `page` parameter asks for a particular one. The loop below asks for page 1, then page 2, and so on, until it has them all, pausing half a second between requests to avoid reaching the rate limits.

In [ ]:
# 🟢 Follow every page
all_records = []
page = 1
while True:
    metadata, records = get_worldbank(endpoint, {"format": "json", "per_page": 50, "page": page},
                                      f"gbr_population_p{page}")
    all_records.extend(records)                 # add this page's records to the list
    if page >= metadata["pages"]:               # stop after the last page
        break
    page += 1
    time.sleep(0.5)

print("Collected", len(all_records), "records; the API says there are", metadata["total"])
check(len(all_records) == metadata["total"], "The loop should collect every record.")

**Professional habit:** compare what you collected with the total the API reports. It is a one-line test that the loop did not stop early.

## A4. From JSON records to a table

`all_records` is a list of dictionaries, and some values are themselves dictionaries. First try the obvious approach:

In [ ]:
# 🟢 The obvious approach
plain = pd.DataFrame(all_records)
plain[["indicator", "country", "date", "value"]].head(3)

**What you should see:** the `indicator` and `country` columns contain whole **dictionaries** in each cell, such as `{'id': 'GB', 'value': 'United Kingdom'}`. You cannot filter or group on those easily. `pd.json_normalize` flattens them into ordinary columns.

> **Method card: `pd.json_normalize(data, record_path=None, meta=None, sep=".", max_level=None)`**
>
> * `data` (**required**): a list of dictionaries (or one dictionary). For a simple list of records, **this is all you need to set.**
> * `sep`: the character joining nested key names. The default `"."` gives columns like `country.value`; setting `sep="_"` gives `country_value`, which is easier to type. **Set this almost always.**
> * `record_path`: use only when each record contains a *list* you want to turn into rows (example in B5). It names the key holding that list.
> * `meta`: use with `record_path`, to keep fields from the outer record next to each inner row.
> * `max_level`: how many levels deep to flatten. Leave it out to flatten everything.
>
> Minimum for today: `pd.json_normalize(records, sep="_")`.

Before flattening, write down what one row will be: **one row is the value of one indicator, for one country, in one year.**

In [ ]:
# 🟢 Flatten, then choose and tidy the columns
pop = pd.json_normalize(all_records, sep="_")
print(pop.columns.tolist())

pop = pop[["country_value", "countryiso3code", "indicator_id", "date", "value"]]
pop["date"] = pop["date"].astype(int)             # text to whole numbers
pop["population_m"] = pop["value"] / 1e6           # millions, easier to read
print(pop.dtypes)
pop.sort_values("date").tail()

**What you should see:** column names such as `indicator_id`, `indicator_value`, `country_id` and `country_value`; after conversion, `date` is an integer and `value` a float. UK population in recent years is about 68 to 69 million. If the latest year has no figure yet, its `value` shows `NaN`.

> **Method card: `series.astype(type)`** converts a column to another type, for example `astype(int)` for whole numbers or `astype(float)`. It fails loudly if a value cannot be converted, which is what you want: better an error than a silent mistake.

## A5. Keep the raw data and a provenance record

Your project must show where each dataset came from, when, and under what licence. Save the records exactly as received, and add one line to a provenance file. World Bank Open Data is published under the **Creative Commons Attribution 4.0** licence (CC BY 4.0): you may reuse it, with attribution/appropriate credits.

In [ ]:
# 🟢 Save raw data and a provenance line
RAW = Path("data/raw"); RAW.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H%M%SZ")      # ISO 8601, in UTC
raw_file = RAW / f"worldbank_GBR_SP.POP.TOTL_{stamp}.json"
raw_file.write_text(json.dumps(all_records), encoding="utf-8")

provenance = {"collected_at": stamp, "source": "World Bank API v2",
              "url": BASE + endpoint, "params": {"format": "json", "per_page": 50},
              "records": len(all_records), "file": raw_file.name,
              "sha256": hashlib.sha256(raw_file.read_bytes()).hexdigest(),
              "licence": "CC BY 4.0 (World Bank Open Data)"}
with open("data/provenance.jsonl", "a", encoding="utf-8") as f:
    f.write(json.dumps(provenance) + "\n")
print(json.dumps(provenance, indent=2))

---
# Part B. Pause and analyse

Work with the person next to you. In every exercise, **paste the printed URL into your browser** to see the raw answer as well.

## B1. Change one piece at a time

The World Bank's code for annual **inflation** (consumer prices, %) is `FP.CPI.TOTL.ZG`. France's three-letter country code is `FRA`.

🟡 Fill the gaps to get France's inflation, **10 records per page**. Then answer the question below the cell.

In [ ]:
country = "FRA"          # a three-letter code, such as GBR, not a country name
indicator = "FP.CPI.TOTL.Z"        # the indicator code
per_page = "10"        # a number: remove the quotes when you replace it

if filled(country, indicator, per_page):
    metadata, records = get_worldbank(f"/country/{country}/indicator/{indicator}",
                                      {"format": "json", "per_page": per_page}, f"{country}_{indicator}_p1")
    fra = pd.json_normalize(records, sep="_")
    print(fra[["country_value", "date", "value"]])
    check(set(fra["countryiso3code"]) == {"FRA"} and set(fra["indicator_id"]) == {"FP.CPI.TOTL.ZG"}
          and len(fra) == 10, "FRA, FP.CPI.TOTL.ZG and 10 per page.")

If you see **World Bank error 120**, a value in the URL is not valid: check that the country is a code (`FRA`) and not a name, and that the indicator code has no typos. Look at the printed URL in your browser to see the error for yourself.

✍️ **Analyse.** With `per_page` set to 10, how many pages does France's inflation series have (`metadata["pages"]`)? Which piece of the URL would you change to get only 2010 to 2019?

*Your answer:*

## B2. Predict the answer from the URL

For each URL, predict `total` and `pages` **before** you run the cell, then check by running it (or by pasting the URL into your browser).

| URL parameters | Your predicted `total` | Your predicted `pages` |
|---|---|---|
| `format=json&per_page=10&date=2010:2019` | | |
| `format=json&per_page=4&date=2010:2019` | | |
| `format=json&per_page=4&date=2019` | | |

In [ ]:
# 🟢 Run after predicting
for p in [{"format": "json", "per_page": 10, "date": "2010:2019"},
          {"format": "json", "per_page": 4, "date": "2010:2019"},
          {"format": "json", "per_page": 4, "date": "2019"}]:
    meta, recs = get_worldbank("/country/FRA/indicator/FP.CPI.TOTL.ZG", p,
                               f"fra_cpi_{p['per_page']}_{p['date'].replace(':', '-')}")
    print(f"   total={meta['total']}, pages={meta['pages']}\n")

**Key output:** 10 records in 1 page; 10 records in 3 pages (4 + 4 + 2); 1 record in 1 page. `date` decides **which** records exist; `per_page` only decides how they are **split**.

## B3. Two APIs, two ways of reporting an error

✍️ Before running the cell, predict: what status code will **Open-Meteo** return for a latitude of 999, which cannot exist? Write it here: ____

In [ ]:
# 🟢 Run after predicting
OM = "https://api.open-meteo.com/v1"
p = {"latitude": 999, "longitude": 0, "current": "temperature_2m"}
show_url(OM, "/forecast", p)
r = requests.get(OM + "/forecast", params=p, headers=HEADERS, timeout=30)
print("Status code:", r.status_code)
print("Body:", r.text[:200])

**Key output:** Open-Meteo returns **400** with a JSON explanation that the latitude is out of range. That is the usual design: the status code alone tells you something is wrong (step 3). The World Bank's 200-with-a-message (A2) is easier to miss, which is why step 4 exists.

✍️ **Reflect (one sentence).** Why is it a professional habit to check the status code *before* calling `r.json()`?

*Your answer:*

## B4. Complete the page loop as a function

🟡 Turn the A3 loop into a function you can reuse for any country and indicator. Three gaps: the `page` parameter, the stopping test, and what to return.

In [ ]:
def fetch_indicator(country, indicator, date=None, per_page=50):
    """All records for a country (or several, joined with ';') and an indicator, following every page."""
    params = {"format": "json", "per_page": per_page}
    if date is not None:
        params["date"] = date
    collected, page = [], 1
    while True:
        meta, recs = get_worldbank(f"/country/{country}/indicator/{indicator}",
                                   {**params, "page": ____}, f"{country}_{indicator}_{date}_{per_page}_p{page}")
        collected.extend(recs)
        if page >= meta["____"]:
            break
        page += 1
        time.sleep(0.5)
    return ____

Several countries can go in one request, separated by `;`. The test asks for **GDP growth** (`NY.GDP.MKTP.KD.ZG`) for the seven G7 countries over 14 years, with only 20 records per page, so the function has to follow 5 pages.

In [ ]:
# Test for B4
G7 = "CAN;FRA;DEU;ITA;JPN;GBR;USA"
growth = fetch_indicator(G7, "NY.GDP.MKTP.KD.ZG", date="2010:2023", per_page=20)
growth_df = pd.json_normalize(growth, sep="_")
check(len(growth_df) == 98, f"Expected 7 countries x 14 years = 98 records, got {len(growth_df)}. Is the page number passed to the request?")

## B5. Practise `json_normalize`'s parameters

Some APIs put a **list inside each record**. Here is a small, invented example: each country record holds a list of yearly readings.

In [ ]:
# 🟢 A small nested example
nested = [
    {"code": "GBR", "name": "United Kingdom", "readings": [{"year": 2022, "value": 9.1}, {"year": 2023, "value": 7.3}]},
    {"code": "FRA", "name": "France",         "readings": [{"year": 2022, "value": 5.2}, {"year": 2023, "value": 4.9}]},
]
print(pd.json_normalize(nested, sep="_"))

**What you should see:** 2 rows, and a `readings` column holding a whole list in each cell. `json_normalize` flattens nested *dictionaries* on its own, but a nested *list* needs `record_path`.

🟡 Use `record_path` to get **one row per reading**, and `meta` to keep each country's `code` and `name` next to its readings.

In [ ]:
long = pd.json_normalize(nested, record_path="____", meta=["____", "____"])
print(long)
check(len(long) == 4 and set(long.columns) == {"year", "value", "code", "name"},
      "record_path names the key holding the list; meta lists the outer fields to keep.")

✍️ **Analyse.** In `pd.json_normalize(nested, sep="_")`, one row is one country. In `long`, what is one row? Which one would you use to compute each country's average, and which to count how many readings there are?

*Your answer:*

## B6. Long or wide?

`growth_df` from B4 is **long**: one row per country–year. Comparing countries side by side is easier **wide**: one row per year, one column per country.

> **Method card: `df.pivot(index, columns, values)`**
>
> * `index` (**required**): the column whose values become the **rows** (here `date`).
> * `columns` (**required**): the column whose values become the **column headings** (here `countryiso3code`).
> * `values` (**required**): the column that fills the table (here `value`).
>
> It fails if two rows share the same index and column, which is a useful check that one row really was one country–year.

🟡 Fill the gaps, then find the country with the lowest growth in 2020.

In [ ]:
growth_df["date"] = growth_df["date"].astype(int)
wide = growth_df.pivot(index="____", columns="____", values="____")
print(wide.round(1).tail())

lowest_2020 = wide.loc[2020].idxmin()     # idxmin: the column with the smallest value in that row
print("Lowest growth in 2020:", lowest_2020)
check(wide.shape == (14, 7), "index='date', columns='countryiso3code', values='value'.")

## B7. A new API: build its URL piece by piece

Open-Meteo has a **historical weather** API. Its documentation is at https://open-meteo.com/en/docs/historical-weather-api. Use it to find the parameter names, then build the request in steps, printing the URL each time and pasting it into your browser.

| Piece | Value |
|---|---|
| Base URL | `https://archive-api.open-meteo.com/v1` |
| Endpoint | `/archive` |
| Parameters you need | `latitude`, `longitude`, `start_date`, `end_date`, `daily`, `timezone` |

🟡 Complete the parameters for **London** (latitude 51.5144, longitude −0.1165), **1 to 31 July 2025**, with the **daily maximum temperature** (`temperature_2m_max`) and times in `Europe/London`.

In [ ]:
ARCHIVE = "https://archive-api.open-meteo.com/v1"
p = {"latitude": 51.5144, "longitude": -0.1165,
     "start_date": "____", "end_date": "____",
     "daily": "____", "timezone": "Europe/London"}

if filled(*p.values()):
    show_url(ARCHIVE, "/archive", p)
    r = requests.get(ARCHIVE + "/archive", params=p, headers=HEADERS, timeout=30)
    print("Status code:", r.status_code)                 # step 3 first
    if r.status_code == 200:
        july = r.json()
        print(list(july.keys()))                          # step 4: what shape came back?
        temps = pd.DataFrame({"date": pd.to_datetime(july["daily"]["time"]),
                              "tmax_c": july["daily"]["temperature_2m_max"]})
        print(temps.describe())
        check(len(temps) == 31, "start_date 2025-07-01, end_date 2025-07-31, daily temperature_2m_max.")

**Key output:** status 200, keys including `daily` and `daily_units`, and a table of **31 days**. Open-Meteo returns **parallel lists** (`"time": [...]` and `"temperature_2m_max": [...]`) rather than a list of records, so `pd.DataFrame` with one column per list is the right tool, and `json_normalize` is not needed. Printing the keys (step 4) is how you find out.

✍️ **Experiment.** Change `daily` to `temperature_2m_max,precipitation_sum` (two variables, separated by a comma). Paste the new URL into your browser. What changes in the answer?

*Your answer:*

---
# Part C. Two APIs, one validated dataset

Real projects rarely use one source. Data is usually collected from multiple sources or systems. For instance, a typical pattern: an **outcome** indicator from one organisation, plus **context and denominators** (population, GDP, income group) from another, joined by similar elements like **country (ISO3 code) and year**. Here you practise it with emissions data, so that you build the project pipeline yourself.

The two sources:

* the **Our World in Data (OWID) chart API**, which serves the data behind every chart on ourworldindata.org, here carbon dioxide (CO₂) emissions from the Global Carbon Budget;
* the **World Bank Indicators API**, as in Parts A and B.

| Stage | What you do | Section |
|---|---|---|
| Collect | One indicator from OWID; matching series from the World Bank | C1, C2 |
| Prepare | Make both tables share the same key columns, with the same types, and one row per key | C1, C2 |
| Join | Merge on `iso3` and `year`, and look at what did **not** match | C3 |
| Validate | Check keys, match rate, ranges, missing values, a spot value, and the right denominator | C4 |

The worked example (C1 to C4) asks: **how do CO₂ emissions per person differ across World Bank income groups, 2015 to 2019?** Then you repeat the pipeline yourself (C5) and turn the checks into a reusable function (C6).

## C1. Worked example: one indicator from the Our World in Data API

### Building the request

Every OWID chart has a page, and the API is that page's address with an ending added:

| Piece | Value | Meaning |
|---|---|---|
| Base URL | `https://ourworldindata.org/grapher` | the chart API |
| Endpoint | `/co-emissions-per-capita.csv` | one chart (its **slug**), as **CSV** data |
| or | `/co-emissions-per-capita.metadata.json` | the same chart's **metadata**: title, units, source, citation |
| Parameters | `v=1`, `csvType=full`, `useColumnShortNames=true` | API version; every country and year (not just what the chart shows); short column names |

To find a slug, search https://ourworldindata.org/data, open a chart, and copy the last part of its address. The helper below builds both URLs and prints them for your browser. [Click here for the full API Documentation.](https://docs.owid.io/projects/etl/api/)

In [ ]:
# 🟢 Build OWID URLs
OWID = "https://ourworldindata.org/grapher"
OWID_PARAMS = {"v": 1, "csvType": "full", "useColumnShortNames": "true"}

def owid_urls(slug):
    """Print and return the data (CSV) and metadata (JSON) URLs for one OWID chart."""
    data_url = show_url(OWID, f"/{slug}.csv", OWID_PARAMS)
    meta_url = show_url(OWID, f"/{slug}.metadata.json", OWID_PARAMS)
    return data_url, meta_url

owid_urls("co-emissions-per-capita")

Open both URLs in your browser. **What you should see:** the first downloads (or displays) a **CSV file**, a format you have not met yet in this lab: plain text, one row per line, values separated by commas. Its first line names the columns: `Entity` (a country or a group), `Code`, `Year`, and one value column with a short name, holding tonnes of CO₂ per person. The next lines look like `Afghanistan,AFG,2019,0.3`. The second URL returns JSON describing the chart: its title, units, and the **citation** you must give.

### Collecting with the professional routine

Same five steps as before. The only difference is step 4: the body is CSV, so we check the content type and read it with `pd.read_csv`.

> **Method card: `pd.read_csv(file)`** reads comma-separated text into a table. To read text you already have in memory (such as `r.text`), wrap it in `StringIO(...)`, which makes text behave like a file.

In [ ]:
# 🟢 The routine for the OWID API
from io import StringIO

def get_owid(slug):
    """Data and metadata for one OWID chart, with status checks and a saved copy of each."""
    data_url, meta_url = owid_urls(slug)
    out = {}
    for kind, url, ext in [("data", data_url, "csv"), ("meta", meta_url, "json")]:
        cache_file = CACHE / f"owid_{slug}.{ext}"
        try:
            r = requests.get(url, headers=HEADERS, timeout=60)
            print(f"{kind}: status {r.status_code}, content type {r.headers.get('Content-Type')}")   # steps 3 and 4
            if r.status_code != 200:
                raise RuntimeError(f"Status {r.status_code}: {r.text[:200]}")
            cache_file.write_bytes(r.content)                          # keep the raw response
            text = r.text
        except requests.RequestException as err:
            if not cache_file.exists():
                raise
            print(f"(Network problem: {type(err).__name__}. Using the saved copy.)")
            text = cache_file.read_text(encoding="utf-8")
        out[kind] = pd.read_csv(StringIO(text)) if ext == "csv" else json.loads(text)
    return out["data"], out["meta"]

co2_raw, co2_meta = get_owid("co-emissions-per-capita")
print("\nTitle:   ", co2_meta["chart"]["title"])
print("Citation:", co2_meta["chart"].get("citation"))
co2_raw.head()

**What you should see:** status 200 twice, content types `text/csv` and `application/json`, the chart's title and citation, and the first rows of the table. Keep the citation: it goes into the provenance record and your report.

### Preparing the OWID table

The table mixes **countries** with **groups**. Look at what has no ordinary three-letter code:

In [ ]:
# 🟢 Look first: entities without an ordinary ISO3 code
co2_raw.columns = co2_raw.columns.str.lower()             # Entity, Code, Year -> entity, code, year
odd = co2_raw[~co2_raw["code"].fillna("").str.fullmatch(r"[A-Z]{3}")]
print(odd.drop_duplicates("entity")[["entity", "code"]].head(15))

**What you should see:** two kinds of non-country rows.

* Groups with **no code at all**: continents such as *Africa* and *Europe*, and groupings such as *High-income countries* or *European Union (27)*.
* Codes beginning **`OWID_`**: OWID's own codes for entities that have no ISO code, such as `OWID_WRL` (World). One of them **is** a country: `OWID_KOS` is Kosovo, which the World Bank codes `XKX`.

So the preparation has four decisions, each written down:

1. **Recode** the one country with a non-ISO code (`OWID_KOS` to `XKX`). The recode table goes into the provenance record.
2. **Keep countries only:** rows whose code is three capital letters.
3. **Rename and keep** the key columns and the value: `iso3`, `year` (integer), `co2_per_capita`, and keep only 2015 to 2019.
4. **One row is one country–year.** Check it.

In [ ]:
# 🟢 Prepare the OWID table
OWID_TO_WB = {"OWID_KOS": "XKX"}                                             # 1. recode
value_col = [c for c in co2_raw.columns if c not in ("entity", "code", "year")]
check(len(value_col) == 1, "Expected exactly one value column.")

co2 = co2_raw.assign(code=co2_raw["code"].replace(OWID_TO_WB))
co2 = co2[co2["code"].fillna("").str.fullmatch(r"[A-Z]{3}")]                 # 2. countries only
co2 = co2.rename(columns={"code": "iso3", value_col[0]: "co2_per_capita"})   # 3. rename
co2 = co2[["iso3", "year", "co2_per_capita"]]
co2["year"] = co2["year"].astype(int)
YEARS = range(2015, 2020)
co2 = co2[co2["year"].isin(YEARS)]

check(not co2.duplicated(["iso3", "year"]).any(), "One row should be one country-year.")    # 4.
print(co2["iso3"].nunique(), "countries,", len(co2), "rows")

**What you should see:** about 220 countries and territories over 5 years, and both checks ✅.

> **Method card: `df.duplicated(columns)`** is True for each row whose values in those columns already appeared above it. If a key is duplicated, a join multiplies rows silently, so this check comes **before** every join.

## C2. Worked example: context and the right denominator from the World Bank

The outcome is CO₂ **per person**, so the right denominator for weighting is the **population** (`SP.POP.TOTL`). We also take **GDP per capita** (`NY.GDP.PCAP.CD`) as context, and each country's **income group**.

`wb_series` is the page loop from A3 as a function, with 1,000 records per page to keep the number of requests small.

In [ ]:
# 🟢 World Bank series for all countries
def wb_series(indicator, date, country="all"):
    """Every record for one World Bank indicator, following every page."""
    collected, page = [], 1
    while True:
        meta, recs = get_worldbank(f"/country/{country}/indicator/{indicator}",
                                   {"format": "json", "per_page": 1000, "date": date, "page": page},
                                   f"wb_{country}_{indicator}_{date.replace(':', '-')}_p{page}")
        collected.extend(recs)
        if page >= meta["pages"]:
            return collected
        page += 1
        time.sleep(0.5)

def wb_table(records, value_name):
    """Records to a table with the shared key names and types."""
    t = pd.json_normalize(records, sep="_")
    t = t.rename(columns={"countryiso3code": "iso3", "date": "year", "value": value_name})
    t["year"] = t["year"].astype(int)                  # the World Bank sends years as text
    return t[["iso3", "year", value_name]]

date = "2015:2019"
pop = wb_table(wb_series("SP.POP.TOTL", date), "population")
gdppc = wb_table(wb_series("NY.GDP.PCAP.CD", date), "gdp_per_capita_usd")
print(len(pop), "population rows;", len(gdppc), "GDP per capita rows")

The World Bank has the same trap as OWID: `country/all` returns **aggregates** (`WLD`, `EUU`, `HIC`...). They have three-letter codes, so the "three capital letters" test does not catch them. Its `/country` endpoint marks them with the region `Aggregates`, and gives each country's income group.

In [ ]:
# 🟢 Countries only, with income group, one row per country-year
meta, country_list = get_worldbank("/country", {"format": "json", "per_page": 400}, "wb_countries")
countries = pd.json_normalize(country_list, sep="_")
countries = countries[countries["region_value"] != "Aggregates"]
countries = countries.rename(columns={"id": "iso3", "name": "country", "incomeLevel_value": "income_group"})
countries = countries[["iso3", "country", "income_group"]]

wb = pop.merge(gdppc, on=["iso3", "year"], how="outer")
wb = wb[wb["iso3"].isin(countries["iso3"])].merge(countries, on="iso3", how="left")
check(not wb.duplicated(["iso3", "year"]).any(), "One row should be one country-year.")
print(wb["iso3"].nunique(), "countries,", len(wb), "rows")
wb.head()

**What you should see:** about 217 countries over 5 years. Both tables now have `iso3` (text) and `year` (integer) with exactly these names. The types matter: `2019` and `"2019"` are **not** equal, so a join on mismatched types finds nothing.

## C3. The join

> **Method card: `pd.merge(left, right, on, how, validate, indicator)`**
>
> * `left`, `right` (**required**): the two tables. Put the **outcome** table on the left.
> * `on` (**required here**): the key columns, with the same names in both: `["iso3", "year"]`.
> * `how`: `"inner"` keeps rows in both; `"left"` keeps every row of `left`; `"outer"` keeps every row of either.
> * `validate="one_to_one"`: raise an error if either table has a duplicated key. **Always set it.**
> * `indicator=True`: adds a `_merge` column, `both`, `left_only` or `right_only`. **Set it while you check a join.**

Start with an **outer** join so you can see what failed to match. The match rate that matters is the share of **outcome rows** (the OWID table) that found a World Bank partner.

In [ ]:
# 🟢 Join, keeping everything, and look at the result
check_join = pd.merge(co2, wb, on=["iso3", "year"], how="outer", validate="one_to_one", indicator=True)
print(check_join["_merge"].value_counts(), "\n")

print("In OWID but not the World Bank:", sorted(check_join.loc[check_join["_merge"] == "left_only", "iso3"].unique()))
print("In the World Bank but not OWID:", sorted(check_join.loc[check_join["_merge"] == "right_only", "iso3"].unique()))
match_rate = (check_join["_merge"] == "both").sum() / len(co2)
print(f"\nShare of OWID rows matched: {match_rate:.1%}")

**What you should see:** most rows `both`, a high match rate, and two short lists. Read them; they are information, not errors:

* **OWID only:** small territories that the World Bank does not list separately, such as `AIA` (Anguilla), `COK` (Cook Islands) or `NIU` (Niue).
* **World Bank only:** places for which the Global Carbon Budget gives no separate figure in these years.

Kosovo is **not** in either list, because we recoded `OWID_KOS` in C1. Every project report should say how many rows matched and why the rest did not. For the analysis we keep the **inner** join.

In [ ]:
# 🟢 The analysis table
merged = check_join[check_join["_merge"] == "both"].drop(columns="_merge")
print(merged.shape)
merged.head()

## C4. Validate the joined data

A join that runs is not a join that is right. Five checks, each a few lines. They are what your project's reproducibility evidence should contain.

In [ ]:
# 🟢 Checks 1 to 3: keys and match rate; plausible ranges; missing values
check(not merged.duplicated(["iso3", "year"]).any(), "No duplicated keys after the join.")
check(match_rate > 0.9, "Fewer OWID rows matched than expected: check key types, names and codes.")

check(merged["co2_per_capita"].between(0, 60).all(), "CO2 per person outside 0-60 tonnes: a unit or code problem?")
check((merged["population"].dropna() > 0).all(), "Population must be positive.")

print("\nMissing values per column:\n", merged.isna().sum())
print("\nRows per year:\n", merged.groupby("year").size())

**What you should see:** all checks ✅, a few missing GDP values (not every country reports every year), and roughly the same number of rows each year. A year with far fewer rows would mean one source has not published it yet.

In [ ]:
# 🟢 Check 4: a spot check against both sources, by hand
print(merged[(merged["iso3"] == "GBR") & (merged["year"] == 2019)].T, "\n")
print("OWID chart (hover over the United Kingdom, 2019): https://ourworldindata.org/grapher/co-emissions-per-capita?country=GBR")
show_url(BASE, "/country/GBR/indicator/SP.POP.TOTL", {"format": "json", "date": "2019"})

Open both in a browser and compare the values with the table. If they match, the whole chain (request, reading the CSV, recode, filter, rename, type conversion, join) worked for at least one known case. Record the spot check in your report.

### Check 5: the right denominator

The average of countries' per-person emissions treats Qatar and China alike. Weighting each country by its **population** gives the emissions of the average **person** in the income group, which is the same as the group's total emissions divided by its total population. Because the outcome is *per person*, population is the right denominator.

In [ ]:
# 🟢 Check 5: unweighted versus population-weighted, by income group (2019)
y = merged[merged["year"] == 2019].dropna(subset=["population"])
summary = y.groupby("income_group").apply(
    lambda g: pd.Series({"countries": len(g),
                         "mean_of_countries": g["co2_per_capita"].mean(),
                         "population_weighted": (g["co2_per_capita"] * g["population"]).sum() / g["population"].sum()}),
    include_groups=False).round(2)
summary

**What you should see:** emissions per person rising steeply with income group, and the two averages differing. In the high-income group, small oil and gas producers with very high emissions per person pull the unweighted mean up; weighting by population brings the value closer to the large economies. A plausible gradient across income groups is also a final sanity check on the join: if low-income countries came out highest, something upstream would be wrong.

### Save with provenance

Save the joined table, and one provenance line per source with its licence, citation and the recode table. Both OWID and the World Bank publish under **CC BY 4.0**; OWID also asks you to cite the **underlying source** (here the Global Carbon Budget), which is the citation in the metadata.

In [ ]:
# 🟢 Save
stamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H%M%SZ")
Path("data/processed").mkdir(parents=True, exist_ok=True)
merged.to_csv("data/processed/co2_population_gdp.csv", index=False)
with open("data/provenance.jsonl", "a", encoding="utf-8") as f:
    for src in [{"source": "Our World in Data chart API", "chart": "co-emissions-per-capita", "years": "2015-2019",
                 "records": len(co2), "licence": "CC BY 4.0", "citation": co2_meta["chart"].get("citation")},
                {"source": "World Bank API v2", "indicator": "SP.POP.TOTL; NY.GDP.PCAP.CD", "years": date,
                 "records": len(pop) + len(gdppc), "licence": "CC BY 4.0"}]:
        f.write(json.dumps({"collected_at": stamp, **src, "join": "inner on iso3, year",
                            "recode": OWID_TO_WB, "match_rate_outcome_rows": round(match_rate, 3)}) + "\n")
print("Saved", len(merged), "rows")

## C5. Your turn: total emissions, and a cross-source check

Repeat the pipeline with a different OWID chart, then use the World Bank denominator to **check one source against the other**.

**Question:** what share of the world's CO₂ emissions came from each World Bank income group in 2020 to 2022, and does dividing OWID's total emissions by World Bank population reproduce OWID's own per-person figures?

1. 🔵 **Collect from OWID.** The chart `annual-co2-emissions-per-country` gives total emissions in tonnes. Look at its two URLs in your browser first, then collect it with `get_owid`.
2. 🔵 **Prepare.** Lower-case the columns, recode `OWID_KOS`, keep three-letter codes, rename to `iso3`, `year` (integer) and `co2_tonnes`, keep 2020 to 2022, and check one row per country–year.
3. 🔵 **Collect from the World Bank.** Population (`SP.POP.TOTL`) for all countries, 2020 to 2022, with `wb_series`; drop aggregates and add income group (reuse `wb_table` and `countries`).
4. 🔵 **Join** with `how="outer"`, `validate="one_to_one"` and `indicator=True`. List the OWID-only codes and keep `both`.
5. 🔵 **Validate:** share of OWID rows matched, no duplicated keys, no negative totals, missing values per column.
6. 🔵 **Cross-source check.** Compute `co2_per_capita_wb = co2_tonnes / population`. Also collect OWID's per-person series for 2020 to 2022 (as in C1), join it on, and compute the **percentage difference** between the two per-person figures. List the five countries with the largest difference.
7. 🔵 **Answer the question:** each income group's share of total emissions in 2022, as a percentage.

Store the final table in `totals` (columns `iso3`, `year`, `co2_tonnes`, `population`, `income_group`, `co2_per_capita_wb`, ...).

In [ ]:
# Your code here, step by step

In [ ]:
import requests
import pandas as pd

# Search World Bank indicators for 'forest area'
search_url = "https://api.worldbank.org/v2/indicator"
params = {
    "q": "forest area",
    "format": "json",
    "per_page": 50
}

r = requests.get(search_url, params=params)
if r.status_code == 200:
    body = r.json()
    if len(body) > 1 and body[1]:
        indicators = pd.json_normalize(body[1])
        display(indicators[['id', 'name', 'sourceNote']].head(10))
    else:
        print("No indicators found.")
else:
    print(f"Request failed: {r.status_code}")

In [ ]:
# Test for C5
check({"iso3", "year", "co2_tonnes", "population", "income_group", "co2_per_capita_wb"}.issubset(totals.columns), "Name the columns as asked.")
check(totals["year"].between(2020, 2022).all() and totals["year"].dtype.kind == "i", "Years 2020-2022, as integers.")
check(not totals.duplicated(["iso3", "year"]).any(), "One row per country-year after the join.")
check(not totals["iso3"].isin(["WLD", "HIC", "LIC", "EUU"]).any() and not totals["iso3"].str.startswith("OWID").any(),
      "Drop aggregates and OWID group codes.")

✍️ **Interpret (three to four sentences).** For most countries, the two per-person figures should agree to within a few per cent. Why are they not identical? (Read OWID's metadata: whose population figures does OWID use?) For the countries with the largest differences, what might explain them? Which is the better choice for your analysis, and what would you write in your report?

*Your answer:*

## C6. Your turn: a reusable, validated join

A project repeats the same join for several indicators. Turn the checks into a function so they run every time.

🔵 Write `validated_join(left, right, keys=("iso3", "year"), min_match=0.9)` that:

1. checks neither table has duplicated keys (raise `ValueError` with a clear message if one does);
2. checks the key columns have the **same types** in both tables (raise `TypeError` if not);
3. runs an outer merge with `validate="one_to_one"` and `indicator=True`;
4. prints the counts of `both`, `left_only` and `right_only`, and the codes that are only on the left;
5. raises `ValueError` if the share of **left** rows that matched is below `min_match`;
6. returns the inner-joined table without the `_merge` column.

The tests try it on a correct join and on three broken ones.

In [ ]:
def validated_join(left, right, keys=("iso3", "year"), min_match=0.9):
    # Your code here
    pass

In [ ]:
# Tests for C6
ok = validated_join(co2, wb)
check(len(ok) == len(merged), "A correct join should give the same rows as C3.")

def raises(exc, f):
    try:
        f(); return False
    except exc:
        return True

check(raises(ValueError, lambda: validated_join(pd.concat([co2, co2.head(1)]), wb)), "Duplicated keys should raise ValueError.")
check(raises(TypeError, lambda: validated_join(co2.assign(year=co2["year"].astype(str)), wb)), "Text years against integer years should raise TypeError.")
check(raises(ValueError, lambda: validated_join(co2.assign(year=co2["year"] + 20), wb)), "A low match rate should raise ValueError.")

### Taking this to your project

Your project follows the same stages with a health outcome from the **WHO Global Health Observatory API**. The pipeline is the same; the API details differ, so read its documentation the way you read OWID's in C1: how the URL is built, how rows are filtered, how results are paged, which dimensions (such as sex or age group) each indicator has, and how countries are distinguished from regions. Then reuse `validated_join` and the five checks.

---
## Before you leave

- [ ] Parts A and B run, and every test shows ✅.
- [ ] You can explain the three pieces of a request, the five professional steps, and how OWID's URLs and CSV response differ from the World Bank's JSON.
- [ ] C1 to C4 run, and you can explain each validation check and why the denominator is population.
- [ ] You have started C5; C6 is done before your project collection starts.
- [ ] **Edit → Clear all outputs**, download the notebook into your personal repository, and commit it with a clear message. Keep `data/raw/` out of Git; commit `data/provenance.jsonl`.